# Day 1 Community Contribution: Website Moodboard

This notebook is my playful spin on the Day 1 website summarizer.

Instead of only summarizing a page, it turns a website into a compact **moodboard**: what it is, who it is for, what vibe it gives off, and a few practical ideas you could act on.

It uses the same core ideas from the lesson:

- load an API key from `.env`
- fetch website text with the course scraper
- build `system` and `user` messages
- call OpenAI
- display the result nicely as Markdown


## 1. Imports

The small scraper helper below keeps this notebook self-contained.

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI


headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """Return the title and main text from a website, truncated for prompting."""
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"

    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""

    return (title + "\n\n" + text)[:2_000]


## 2. Connect to OpenAI

Same pattern as Day 1: load `.env`, check that the API key exists, then create the client.

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key was found. Add OPENAI_API_KEY to your .env file.")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it does not start with sk-proj-. Please check it.")
elif api_key.strip() != api_key:
    print("The API key has extra whitespace at the start or end. Please clean it up.")
else:
    print("API key found and ready.")

openai = OpenAI()

## 3. Pick a website

Try changing the URL. Simple static sites usually scrape best with the Day 1 scraper.

In [ ]:
url = "https://edwarddonner.com"
website_text = fetch_website_contents(url)

print(website_text[:700])

## 4. Design the prompts

The system prompt gives the model a personality and a job. The user prompt gives it the website contents and the exact output format.

In [ ]:
system_prompt = """
You are a sharp but friendly website analyst.
Your job is to turn raw website text into a useful, fun, business-friendly moodboard.
Be concise, specific, and playful without being mean.
Respond in Markdown only. Do not wrap your response in a code block.
"""


def user_prompt_for(url, website_text):
    return f"""
Analyze this website: {url}

Here is the scraped website text:

{website_text}

Create a Website Moodboard with exactly these sections:

## Snapshot
A 2 sentence summary of what this website is about.

## Vibe Check
Three bullets describing the website's tone, audience, and energy.

## Best Bits
Three bullets naming the strongest parts of the site.

## Tiny Glow-Up Ideas
Three practical ideas that could improve the site or its messaging.

## One-Line Trailer
A fun one-liner that makes the site sound like a movie trailer.
"""

## 5. Build messages

This is the same message structure from the lesson: a `system` message plus a `user` message.

In [ ]:
def messages_for(url, website_text):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(url, website_text)},
    ]


messages_for(url, website_text)

## 6. Create the moodboard

In [ ]:
def create_website_moodboard(url, model="gpt-4.1-mini"):
    website_text = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model=model,
        messages=messages_for(url, website_text),
    )
    return response.choices[0].message.content


def display_website_moodboard(url):
    moodboard = create_website_moodboard(url)
    display(Markdown(moodboard))

In [ ]:
display_website_moodboard("https://edwarddonner.com")

## 7. Bonus: compare a few sites

This turns the Day 1 idea into a tiny research helper. Add or remove URLs in the list below.

In [ ]:
def compare_websites(urls):
    for website_url in urls:
        display(Markdown(f"# Moodboard for {website_url}"))
        display_website_moodboard(website_url)


sample_urls = [
    "https://edwarddonner.com",
    "https://anthropic.com",
]

# Uncomment when you want to run both.
# compare_websites(sample_urls)

# Mini Commercial Example: Inbox DJ

Day 1 suggested trying a commercial summarization idea, like generating email subject lines.

Here is my version: **Inbox DJ**, a tiny tool that turns long emails into subject lines with different styles.

In [ ]:
subject_system_prompt = """
You are Inbox DJ, an assistant that writes excellent email subject lines.
Create subject lines that are clear, short, and useful.
Avoid clickbait. Keep each subject line under 9 words.
Respond in Markdown only.
"""


def subject_line_prompt(email_text):
    return f"""
Here is an email that needs better subject lines:

{email_text}

Suggest 5 subject lines in this format:

| Mode | Subject line |
|---|---|
| Straightforward | ... |
| Warm | ... |
| Executive | ... |
| Energetic | ... |
| Slightly Fun | ... |
"""


def suggest_subject_lines(email_text, model="gpt-4.1-mini"):
    messages = [
        {"role": "system", "content": subject_system_prompt},
        {"role": "user", "content": subject_line_prompt(email_text)},
    ]
    response = openai.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content

In [ ]:
sample_email = """
Hi team,

Quick update before tomorrow's product sync. The landing page copy is ready,
the design review is complete, and the only remaining blocker is final approval
on the pricing table. If we can confirm that by 3 PM, we should still be able
to launch the campaign on Friday morning.

Could everyone please review the pricing notes in the shared doc and leave any
comments before lunch?

Thanks,
Rohit
"""

display(Markdown(suggest_subject_lines(sample_email)))

## What I practiced

- Prompting with clear output sections
- Reusing the `messages = [{"role": ...}]` format
- Building small helper functions around an LLM call
- Making a summarization feature feel more useful and more fun
